# Lab 3.1. JSON: the language APIs speak

**Module 2, Session 3. ACS-UPM Diploma in Engineering, Data Science and Artificial Intelligence**

By the end of this notebook you should be able to:

1. Repair a JSON document from the parser's error messages, and tell valid from correct.
2. Turn a nested response into a flat table with `json_normalize`, keeping the provenance fields.
3. Detect a silent missing value, and write the JSON Schema that rejects it aloud.

Fifteen minutes. Statement, code gap, check cell. **Predict before you run** where marked.

## 0. Setup

This cell locates the course folder wherever the notebook runs: locally, in Colab from the
course repository, or in Colab from the shared Drive folder. Run it first.

In [ ]:
from pathlib import Path

REPO = "https://github.com/antiafer/acs-upm-mod2-s01.git"   # course repository
DRIVE = "ACS-UPM/Mod2-S01"                                 # folder inside My Drive

def course_folder():
    """Return the folder that contains data/, wherever we are running."""
    here = Path.cwd()
    for base in (here, here.parent):
        if (base / "data").is_dir():
            return base
    try:
        import google.colab                            # noqa: F401
    except ImportError:
        raise FileNotFoundError("No data/ folder next to the notebook or one level up.")
    root = Path("/content/acs-mod2")
    if not (root / "data").is_dir():
        import subprocess
        subprocess.run(["git", "clone", "-q", REPO, str(root)], check=False)
    if (root / "data").is_dir():
        return root
    from google.colab import drive
    if not Path("/content/drive").exists():
        drive.mount("/content/drive")
    root = Path("/content/drive/MyDrive") / DRIVE
    if (root / "data").is_dir():
        return root
    raise FileNotFoundError("Could not find the course folder.")

BASE = course_folder()
DATA = BASE / "data"
WORK = Path("/content") if Path("/content").exists() else Path.cwd()
RAW = WORK / "raw"; RAW.mkdir(exist_ok=True)
print("Course folder:", BASE)
print("Raw copies go to:", RAW)

In [ ]:
import importlib, subprocess, sys

def ensure(pkg, module=None):
    try:
        importlib.import_module(module or pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

for pkg, mod in [("requests", None), ("jsonschema", None), ("beautifulsoup4", "bs4"),
                 ("lxml", None), ("pyarrow", None)]:
    ensure(pkg, mod)

import json, re, hashlib, time, datetime as dt
import pandas as pd, requests
print("pandas", pd.__version__, "| requests", requests.__version__)

### Live or cached

Every call in this notebook first tries the live service. If the service does not answer
(twenty people share one address in the room, and public services rate-limit), the cell loads a
copy of the same response saved in `data/`, and says so. The raw copy is not a fallback only: it
is the first step of the pipeline, and the reason a result can be reproduced next year.

**The cached copies are synthetic.** They reproduce the exact shape of each service's answer
(envelope, field names, numbers as strings, missing keys) but the values are invented for the
exercise. Read any number you compute from them as an exercise result, not as a fact about Madrid.
`SOURCES_session3.md` lists what is faithful and what is not in each file.


In [ ]:
def fetch_or_cache(url, cache, *, headers=None, params=None, timeout=15):
    """GET a URL; on any failure, load the cached response from data/. Returns (json, source)."""
    try:
        r = requests.get(url, headers=headers, params=params, timeout=timeout)
        r.raise_for_status()
        doc = r.json()
        # HTTP 200 does not mean the call worked. AEMET answers 200 and puts the real
        # status inside the body, so an expired download link arrives as a normal
        # response reading {"descripcion": "datos expirados", "estado": 404}. Without
        # this check the error envelope flows on as if it were the data.
        if isinstance(doc, dict) and doc.get("estado") not in (None, 200):
            raise ValueError(f"envelope reports estado {doc['estado']}: {doc.get('descripcion')}")
        stamp = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        (RAW / f"{cache.stem}_{stamp}.json").write_text(json.dumps(doc, ensure_ascii=False), encoding="utf-8")
        return doc, "live"
    except Exception as e:                            # noqa: BLE001
        print(f"Live call failed ({type(e).__name__}); using the cached copy {cache.name}")
        return json.loads(cache.read_text(encoding="utf-8")), "cache"

print("fetch_or_cache ready")

## 1. Valid, then correct

### Exercise 1. Repair

The document below arrived from a sensor gateway. It does not parse. Read the error message that
`json.loads` gives you, fix one problem at a time, and store the parsed result in `doc`.

In [ ]:
broken = """
{
  "id": "sensor-P2",
  "status": operational,
  "readings": [
    {"t": 34.1, "unit": "degC"}
    {"t": 35.0, "unit": "degC"}
  ],
}
"""
try:
    json.loads(broken)
except json.JSONDecodeError as e:
    print("Parser says:", e)

In [ ]:
repaired = """
{
  "id": "sensor-P2",
  "status": "operational",
  "readings": [
    {"t": 34.1, "unit": "degC"},
    {"t": 35.0, "unit": "degC"}
  ]
}
"""
doc = json.loads(repaired)
doc

In [ ]:
assert doc["status"] == "operational" and len(doc["readings"]) == 2
assert isinstance(doc["readings"][0]["t"], float), "t must be a number, not a string"
print("Checks passed: three errors fixed (unquoted value, missing comma, trailing comma).")

### Exercise 2. Two things Python will not tell you

JSON has no `NaN`. Python's `json.dumps` writes it anyway, and every other parser rejects the
result. And a missing key can fail at once or fail later, depending on how you ask.
Run the two cells, then answer in the last one.

In [ ]:
import math
sample = {"id": 7, "value": math.nan}
print("dumps by default   :", json.dumps(sample))
try:
    json.dumps(sample, allow_nan=False)
except ValueError as e:
    print("with allow_nan=False:", e)
print("what travels        :", json.dumps({"id": 7, "value": None}))

In [ ]:
reading = {"id": "sensor-P2", "inspection": {"last": "2026-03-12"}}
print("get      :", reading.get("owner"))
try:
    reading["inspection"]["rating"]
except KeyError as e:
    print("brackets : KeyError", e)

In [ ]:
# In one line each: which accessor fails at once, and which one fails later or never?
answer = {"fails_at_once": "doc['key']", "fails_later_or_never": "doc.get('key')"}
assert answer["fails_at_once"] != answer["fails_later_or_never"]
print("Recorded. Choose the accessor deliberately: a loud failure is cheaper than a silent NaN.")

## 2. From nested to flat

An API answer is a tree; an analysis wants a table. `pandas.json_normalize` walks the tree.
Its two arguments that matter are `record_path`, the list to become rows, and `meta`, the fields
higher up the tree to copy onto every row.

### Exercise 3. Bridges from Overpass

`overpass_bridges_madrid.json` is the response of the Overpass API to a query for bridges in
central Madrid. Look at its shape first, then flatten `elements` into a DataFrame `bridges`.
Nested tags become columns named `tags.name`, `tags.highway` and so on.

In [ ]:
resp = json.loads((DATA / "overpass_bridges_madrid.json").read_text(encoding="utf-8"))
print(list(resp.keys()))
print(json.dumps(resp["elements"][0], indent=1, ensure_ascii=False)[:400])

In [ ]:
bridges = pd.json_normalize(resp["elements"])
print(bridges.shape)
bridges[["id", "type", "tags.bridge", "tags.highway", "tags.name"]].head()

In [ ]:
assert len(bridges) == 47, f"Expected 47 bridges, got {len(bridges)}"
assert "tags.name" in bridges.columns, "Nested tags should have become dotted columns"
assert bridges["tags.name"].notna().sum() == 12
print("Checks passed: 47 bridges, 12 of them named. Unnamed ways are normal: OSM tags are optional.")

### Exercise 4. Stations, with provenance on every row

`bicimad_stations.json` is a station-status feed: a header with the timestamp of the snapshot and
a `data` list of stations, each with nested `geometry` and `dock` objects.

Flatten it so that each station is a row **and the feed's `datetime` is copied onto every row**.
That timestamp is the provenance of the snapshot; without it, a row of bike counts means nothing.

In [ ]:
feed = json.loads((DATA / "bicimad_stations.json").read_text(encoding="utf-8"))
print({k: (v if k != "data" else f"list of {len(v)}") for k, v in feed.items()})

In [ ]:
stations = pd.json_normalize(feed, record_path="data", meta=["datetime"])
print(stations.shape)
stations[["id", "name", "dock.total_bases", "dock.dock_bikes", "datetime"]].head(3)

In [ ]:
assert len(stations) == 40
assert "datetime" in stations.columns and stations["datetime"].nunique() == 1
assert "dock.dock_bikes" in stations.columns
print("Checks passed: 40 stations, each row carrying the snapshot time.")

**Predict before you run:** the feed has 40 stations. How many values will `dock.dock_bikes` have?

In [ ]:
missing = stations["dock.dock_bikes"].isna().sum()
print("Stations without a bike count:", missing)
print(stations.loc[stations["dock.dock_bikes"].isna(), ["id", "name", "no_available"]])

One station arrived without its `dock` block. `json_normalize` did exactly what it should:
it produced a NaN and said nothing. The mean of `dock_bikes` is now computed over 39 stations,
and no error tells you so. This is the silent failure of JSON, and the reason for the next exercise.

## 3. The contract

### Exercise 5. A schema that rejects the silent NaN

JSON Schema describes what a document must look like: which keys are required, which types are
allowed. Write `station_schema` so that a station **must** have `id`, `name` and a `dock` object,
and `dock` must have the three integer counts. Validate every station; collect the ids that fail.

In [ ]:
from jsonschema import validate, ValidationError

station_schema = {
    "type": "object",
    "required": ["id", "name", "dock"],
    "properties": {
        "id": {"type": "integer"},
        "name": {"type": "string"},
        "dock": {
            "type": "object",
            "required": ["total_bases", "dock_bikes", "free_bases"],
            "properties": {k: {"type": "integer", "minimum": 0}
                           for k in ("total_bases", "dock_bikes", "free_bases")},
        },
    },
}

rejected = []
for st in feed["data"]:
    try:
        validate(instance=st, schema=station_schema)
    except ValidationError as e:
        rejected.append((st["id"], e.message))
print("Rejected:", rejected)

In [ ]:
assert len(rejected) == 1 and rejected[0][0] == 17, "Exactly one station should fail: id 17"
print("Checks passed: the schema says out loud what json_normalize kept silent.")

> **Valid is not correct.** The feed was valid JSON throughout. The schema is the line between
> *the parser accepted it* and *the analysis can use it*. In Lab 3.2 a schema guards the door of
> the pipeline: nothing enters the typed table without passing it.

### Extension A. JSON Lines

Large dumps are not one JSON array but one JSON object per line, `.jsonl`, so that a file can be
read and written in a stream. Write the 40 stations as JSON Lines and read them back with pandas.

In [ ]:
path = WORK / "stations.jsonl"
with open(path, "w", encoding="utf-8") as f:
    for st in feed["data"]:
        f.write(json.dumps(st, ensure_ascii=False) + "\n")
back = pd.read_json(path, lines=True)
assert len(back) == 40
print("Checks passed:", path.name, "has one station per line.")

---

## Take-aways

- The parser's error message tells you where a document breaks; fix one problem at a time.
- Python writes NaN into JSON and reads absent keys as None; both are silent. Missing values travel as `null`.
- `json_normalize(doc, record_path=..., meta=...)` turns a tree into rows and keeps provenance on each.
- A missing block becomes a NaN without a word. A JSON Schema turns that silence into a rejection.

**References.** IETF RFC 8259 (The JSON Data Interchange Format). JSON Schema 2020-12.
The JSON block follows course material by D. Garijo and P. Calleja (UPM).